# Dịch VI → EN bằng VinAI Translate

**Đầu vào:** `metadata.csv` (cột `id`, `vi_text`) trên Google Drive. Notebook này **không cần audio**.
**Đầu ra:** `metadata_with_en_vinai.csv` (thêm cột `en_text`), cùng cấu trúc với file của notebook NLLB.

Cơ chế an toàn khi Colab/máy dừng đột ngột (giữ nguyên như notebook NLLB):
- Mỗi batch dịch xong được **append vào `translations_vinai.jsonl`** và `fsync` ngay. Mất tối đa 1 batch đang dịch.
- Chạy lại là tự bỏ qua các `id` đã dịch.
- CSV cuối chỉ được gộp **một lần ở cell cuối** (ghi atomic).
- Hết VRAM: tự chia đôi batch và thử lại. Câu nào vẫn lỗi được ghi vào `translate_failed_vinai.jsonl`.
- RAM guard: RAM trống thấp thì dừng êm, tiến độ đã lưu.

**Khác với notebook NLLB:** mô hình `vinai/vinai-translate-vi2en-v2`, mã ngôn ngữ nguồn `vi_VN`, ép token đích `en_XX` bằng `decoder_start_token_id`. Các file checkpoint và đầu ra có hậu tố `_vinai` để không ghi đè kết quả NLLB.

Giấy phép VinAI Translate là AGPL-3.0 (theo trang GitHub của VinAI). `en_text` là pseudo-label do máy dịch, không phải bản dịch tham chiếu do người kiểm chứng.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Cài thư viện
Chạy xong, nếu Colab yêu cầu thì **Runtime > Restart session**, rồi chạy tiếp từ cell 3.

In [2]:
!pip -q install -U "transformers<5" sentencepiece accelerate pandas psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you h

## 3. Cấu hình

In [3]:
import os, gc, json, time
import pandas as pd
import psutil
import torch

# --- Sửa đường dẫn cho đúng thư mục chứa metadata.csv trên Drive ---
DATA_DIR = "/content/drive/MyDrive/project_DPL/datasets/vietspeech_100k"
MODEL_DIR = "/content/drive/MyDrive/project_DPL/models/vinai-translate-vi2en-v2"

INPUT_CSV = f"{DATA_DIR}/metadata.csv"
OUTPUT_CSV = f"{DATA_DIR}/metadata_with_en_vinai.csv"
CKPT_PATH = f"{DATA_DIR}/translations_vinai.jsonl"        # append-only checkpoint
FAILED_PATH = f"{DATA_DIR}/translate_failed_vinai.jsonl"

MODEL_NAME = "vinai/vinai-translate-vi2en-v2"
SRC_LANG, TGT_LANG = "vi_VN", "en_XX"

BATCH_SIZE = 16          # tự chia đôi nếu hết VRAM
NUM_BEAMS = 4
MAX_INPUT_TOKENS = 512
MAX_OUTPUT_TOKENS = 256
USE_FP16 = True          # nếu đầu ra rỗng hoặc lạ, đặt False
MIN_AVAILABLE_GB = 1.0   # RAM trống dưới ngưỡng này thì dừng êm
STOP_AFTER_N = 100      # đặt số (vd 200) để chạy thử một đoạn, None = chạy hết

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (rất chậm)")
assert os.path.isfile(INPUT_CSV), f"Không thấy {INPUT_CSV}"

def log_memory(tag=""):
    r = psutil.virtual_memory()
    s = f"[{tag}] RAM used={r.percent}% avail={r.available/1024**3:.2f}GB"
    if device == "cuda":
        s += f" | GPU alloc={torch.cuda.memory_allocated()/1024**3:.2f}GB"
    print(s)

Device: cpu | CPU (rất chậm)


## 4. Load model (cache lên Drive)

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Kiểm tra xem mô hình đã được cache trên Drive chưa
has_cache = os.path.isfile(os.path.join(MODEL_DIR, "config.json"))
load_from = MODEL_DIR if has_cache else MODEL_NAME
print("Loading from:", load_from)

tokenizer = AutoTokenizer.from_pretrained(load_from, src_lang=SRC_LANG)
dtype = torch.float16 if (device == "cuda" and USE_FP16) else torch.float32

try:
    model = AutoModelForSeq2SeqLM.from_pretrained(load_from, dtype=dtype, low_cpu_mem_usage=True)
except TypeError:
    model = AutoModelForSeq2SeqLM.from_pretrained(load_from, torch_dtype=dtype, low_cpu_mem_usage=True)

model = model.to(device)
model.eval()

# Nếu chưa có sẵn trên Drive, tiến hành tạo thư mục và lưu lại
if not has_cache:
    print(f"Thư mục cache chưa tồn tại hoặc thiếu config. Tiến hành tạo thư mục và lưu model vào: {MODEL_DIR}")
    os.makedirs(MODEL_DIR, exist_ok=True)
    tokenizer.save_pretrained(MODEL_DIR)
    model.save_pretrained(MODEL_DIR, safe_serialization=True)
    print("Đã lưu thành công model và tokenizer vào Drive!")
else:
    print("Model đã có sẵn trên Drive, không cần lưu lại.")

TGT_ID = tokenizer.convert_tokens_to_ids(TGT_LANG)
if TGT_ID is None or TGT_ID == tokenizer.unk_token_id:
    TGT_ID = tokenizer.lang_code_to_id[TGT_LANG]
assert TGT_ID is not None and TGT_ID != tokenizer.unk_token_id, "Không tìm thấy mã ngôn ngữ đích en_XX"

gc.collect()
log_memory("model ready")

Loading from: vinai/vinai-translate-vi2en-v2


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.41M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.69G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.69G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Thư mục cache chưa tồn tại hoặc thiếu config. Tiến hành tạo thư mục và lưu model vào: /content/drive/MyDrive/project_DPL/models/vinai-translate-vi2en-v2


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 1024, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Đã lưu thành công model và tokenizer vào Drive!
[model ready] RAM used=27.4% avail=9.20GB


## 5. Hàm dịch (tự chia đôi batch khi OOM)

In [ ]:
def _translate(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                       max_length=MAX_INPUT_TOKENS).to(device)
    with torch.inference_mode():
        out = model.generate(**inputs, decoder_start_token_id=TGT_ID,
                             num_beams=NUM_BEAMS, early_stopping=True,
                             max_new_tokens=MAX_OUTPUT_TOKENS)
    res = [t.strip() for t in tokenizer.batch_decode(out, skip_special_tokens=True)]
    del inputs, out
    return res

def translate_adaptive(texts):
    try:
        return _translate(texts)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(texts) == 1:
            raise
        mid = len(texts) // 2
        return translate_adaptive(texts[:mid]) + translate_adaptive(texts[mid:])

# Smoke test
print(translate_adaptive(["Tôi đang học trí tuệ nhân tạo.", "Hôm nay thời tiết rất đẹp."]))

["I'm studying artificial intelligence.", "It's a beautiful day."]


## 6. Đọc dữ liệu + checkpoint

In [ ]:
def read_jsonl(path):
    items = []
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    items.append(json.loads(line))
                except json.JSONDecodeError:
                    pass   # dòng cuối có thể bị cắt dở khi session bị kill
    return items

def append_jsonl(path, items):
    with open(path, "a", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())

df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False, encoding="utf-8-sig")
for col in ("id", "vi_text"):
    assert col in df.columns, f"Thiếu cột {col}. Có: {list(df.columns)}"

n_raw = len(df)
df = df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
print(f"Dòng trong CSV: {n_raw:,} | sau khi bỏ id trùng: {len(df):,}")

done = {it["id"] for it in read_jsonl(CKPT_PATH) if it.get("en_text")}
vi = dict(zip(df["id"], df["vi_text"].str.strip()))
empty_vi = [i for i, t in vi.items() if not t]

pending = [i for i, t in vi.items() if t and i not in done]
pending.sort(key=lambda i: len(vi[i]), reverse=True)   # câu dài trước, ít padding, lộ OOM sớm

# Tính toán số lượng thực tế sẽ chạy dựa trên STOP_AFTER_N
will_run = min(len(pending), STOP_AFTER_N) if STOP_AFTER_N else len(pending)

print(f"Đã dịch: {len(done):,} | Tổng cần dịch còn lại: {len(pending):,} | Sẽ dịch lượt này: {will_run:,} (STOP_AFTER_N={STOP_AFTER_N}) | Transcript rỗng: {len(empty_vi):,}")

Dòng trong CSV: 100,000 | sau khi bỏ id trùng: 100,000
Đã dịch: 296 | Tổng cần dịch còn lại: 99,704 | Sẽ dịch lượt này: 99,704 (STOP_AFTER_N=None) | Transcript rỗng: 0


## 7. Dịch (chạy lại cell này để resume)

In [ ]:
t0 = time.time()
translated_now, stopped_reason = 0, None
todo = pending[:STOP_AFTER_N] if STOP_AFTER_N else pending

try:
    for start in range(0, len(todo), BATCH_SIZE):
        ids = todo[start:start + BATCH_SIZE]
        texts = [vi[i] for i in ids]
        try:
            outs = translate_adaptive(texts)
            append_jsonl(CKPT_PATH, [{"id": i, "en_text": o} for i, o in zip(ids, outs)])
            translated_now += len(ids)
        except Exception as e:
            # Thử từng câu để chỉ loại câu thực sự lỗi
            gc.collect()
            if device == "cuda": torch.cuda.empty_cache()
            ok_items, bad_items = [], []
            for i in ids:
                try:
                    ok_items.append({"id": i, "en_text": translate_adaptive([vi[i]])[0]})
                except Exception as e2:
                    bad_items.append({"id": i, "error": repr(e2)})
            if ok_items:  append_jsonl(CKPT_PATH, ok_items)
            if bad_items: append_jsonl(FAILED_PATH, bad_items)
            translated_now += len(ok_items)
            print(f"Batch lỗi ({e!r}); lưu {len(ok_items)}, bỏ {len(bad_items)}")

        n = start + len(ids)
        if (start // BATCH_SIZE) % 20 == 0:
            rate = translated_now / max(time.time() - t0, 1e-9)
            print(f"{n:,}/{len(todo):,} | {rate:.1f} câu/s")
            log_memory("progress")

        if psutil.virtual_memory().available / 1024**3 < MIN_AVAILABLE_GB:
            stopped_reason = "RAM thấp"
            break
except KeyboardInterrupt:
    stopped_reason = "bị ngắt thủ công"

log_memory("end")
print(f"Dịch thêm {translated_now:,} câu trong {(time.time()-t0)/60:.1f} phút.")
if stopped_reason:
    print(f"DỪNG SỚM ({stopped_reason}). Tiến độ đã lưu. Restart session rồi chạy lại từ cell 3 để resume.")

16/99,704 | 18.3 câu/s
[progress] RAM used=47.1% avail=6.71GB | GPU alloc=0.80GB
336/99,704 | 17.4 câu/s
[progress] RAM used=47.1% avail=6.70GB | GPU alloc=0.80GB
656/99,704 | 17.4 câu/s
[progress] RAM used=47.1% avail=6.70GB | GPU alloc=0.80GB
976/99,704 | 17.9 câu/s
[progress] RAM used=47.1% avail=6.70GB | GPU alloc=0.80GB
1,296/99,704 | 18.0 câu/s
[progress] RAM used=47.2% avail=6.69GB | GPU alloc=0.80GB
1,616/99,704 | 18.3 câu/s
[progress] RAM used=47.2% avail=6.69GB | GPU alloc=0.80GB
1,936/99,704 | 17.6 câu/s
[progress] RAM used=47.2% avail=6.69GB | GPU alloc=0.80GB
2,256/99,704 | 17.9 câu/s
[progress] RAM used=47.2% avail=6.69GB | GPU alloc=0.80GB
2,576/99,704 | 18.2 câu/s
[progress] RAM used=47.2% avail=6.70GB | GPU alloc=0.80GB
2,896/99,704 | 18.5 câu/s
[progress] RAM used=47.2% avail=6.70GB | GPU alloc=0.80GB
3,216/99,704 | 18.7 câu/s
[progress] RAM used=47.2% avail=6.69GB | GPU alloc=0.80GB
3,536/99,704 | 18.8 câu/s
[progress] RAM used=47.2% avail=6.70GB | GPU alloc=0.80GB
3

## 8. Gộp thành `metadata_with_en_vinai.csv` (chạy được cả khi mới dịch xong một phần)

In [ ]:
trans = {}
for it in read_jsonl(CKPT_PATH):
    if it.get("en_text"):
        trans[it["id"]] = it["en_text"]          # nếu trùng, bản sau cùng thắng

out = df.copy()
out["en_text"] = out["id"].map(trans).fillna("")

tmp = OUTPUT_CSV + ".tmp"
out.to_csv(tmp, index=False, encoding="utf-8-sig")
os.replace(tmp, OUTPUT_CSV)

n_vi = (out["vi_text"].str.strip() != "").sum()
n_en = (out["en_text"] != "").sum()
print(f"Đã ghi {OUTPUT_CSV}")
print(f"Có en_text: {n_en:,}/{n_vi:,} dòng có transcript | còn thiếu: {n_vi - n_en:,}")
print("Lưu ý: en_text là pseudo-label do máy dịch.")

Đã ghi /content/drive/MyDrive/project_DPL/datasets/vietspeech_100k/metadata_with_en_vinai.csv
Có en_text: 100,000/100,000 dòng có transcript | còn thiếu: 0
Lưu ý: en_text là pseudo-label do máy dịch.


## 9. Xem thử ngẫu nhiên để kiểm tra chất lượng

In [ ]:
chk = out[out["en_text"] != ""]
samples = chk.sample(min(10, len(chk)), random_state=42)[["id", "vi_text", "en_text"]]

for _, row in samples.iterrows():
    print(f"ID: {row['id']}")
    print(f"vi: {row['vi_text']}")
    print(f"en: {row['en_text']}")
    print("======================")

ID: vs_075721
vi: và nó là sức mạnh của linh hồn bạn
en: and it is the power of your soul
ID: vs_080184
vi: cả đêm tôi trằn trọc lo không chợp được mắt
en: I've been tossing and turning all night, worried about not getting my eyes shut.
ID: vs_019864
vi: chương hai thực hành thiền quán
en: Chapter Two: Meditation Practice
ID: vs_076699
vi: còn mọi việc trong nhà đùn hết cho bác
en: Everything else in the house is extruded for you.
ID: vs_092991
vi: và nhận thấy sức khỏe tinh thần của mình có dấu hiệu đi xuống
en: and found that his mental health was showing signs of deteriorating
ID: vs_076434
vi: giờ vua nước cử lại tham lễ vật của nước lỗ mới sai người bảo khánh phủ rằng
en: Now the king of the kingdom sent again to the offering of the kingdom of Hole, and sent him to tell the kingdom that
ID: vs_084004
vi: con muốn ngủ rồi
en: I want to sleep.
ID: vs_080917
vi: tôi đã đích thân tìm xem có cách gì tát cạn đường hào hay không nhưng thấy không thể được
en: I personally looked for a way